# Data exploration

Measurements behind the preprocessing choices: data quality, labels, lengths, hypothesis differences, vocabulary and embedding coverage. Text statistics use the training split, except where a check compares splits.

In [1]:
import collections, html, io, re, string, sys, zipfile
sys.path.append("..")

import datasets
import nltk
import numpy as np
from huggingface_hub.utils import logging as hf_logging
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from src.paths import EMBEDDINGS_DIR

datasets.logging.set_verbosity_error()
datasets.disable_progress_bars()
hf_logging.set_verbosity_error()
nltk.download("punkt_tab", quiet=True)

ds = datasets.load_dataset("amarfurt/art-nlp-hs26")
FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]
PUNCT = set(string.punctuation + "’‘“”—–…")


def is_punct(token):
    return all(c in PUNCT for c in token)


print({split: len(ds[split]) for split in ds}, ds["train"].column_names)
print(ds["train"][0])

{'train': 16801, 'validation': 1000, 'test': 1532} ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


## Data quality

Is format cleaning needed? Do the splits overlap?

In [2]:
for split in ds:
    rows = list(ds[split])
    texts = [r[f] for r in rows for f in FIELDS]
    print(split, {
        "empty": sum(not t.strip() for t in texts),
        "html_tags": sum(bool(re.search(r"</?[a-zA-Z][^>]*>", t)) for t in texts),
        "html_entities": sum(html.unescape(t) != t for t in texts),
        "urls": sum(bool(re.search(r"https?://|www\.", t)) for t in texts),
        "outer_whitespace": sum(t != t.strip() for t in texts),
        "double_spaces": sum("  " in t for t in texts),
        "non_ascii": sum(not t.isascii() for t in texts),
    })

chars = collections.Counter(c for r in ds["train"] for f in FIELDS for c in r[f] if not c.isascii())
print("non-ASCII characters in train:", chars.most_common(15))

pairs = {s: {(r["observation_1"], r["observation_2"]) for r in ds[s]} for s in ds}
print("observation pairs per split:", {s: len(p) for s, p in pairs.items()})
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    print(f"pairs shared by {a} and {b}:", len(pairs[a] & pairs[b]))
same = {s: sum(r["hypothesis_1"] == r["hypothesis_2"] for r in ds[s]) for s in ds}
print("rows with identical hypotheses:", same)

train {'empty': 0, 'html_tags': 0, 'html_entities': 0, 'urls': 0, 'outer_whitespace': 0, 'double_spaces': 68, 'non_ascii': 38}
validation {'empty': 0, 'html_tags': 0, 'html_entities': 0, 'urls': 0, 'outer_whitespace': 0, 'double_spaces': 5, 'non_ascii': 0}
test {'empty': 0, 'html_tags': 0, 'html_entities': 0, 'urls': 0, 'outer_whitespace': 0, 'double_spaces': 6, 'non_ascii': 2}


non-ASCII characters in train: [('’', 32), ('�', 8), ('é', 2), ('´', 1)]


observation pairs per split: {'train': 16801, 'validation': 1000, 'test': 1532}
pairs shared by train and validation: 0
pairs shared by train and test: 0
pairs shared by validation and test: 0


rows with identical hypotheses: {'train': 8, 'validation': 1, 'test': 0}


## Labels

The chance baseline.

In [3]:
for split in ds:
    counts = collections.Counter(ds[split]["label"])
    total = len(ds[split])
    print(split, dict(sorted(counts.items())), {k: round(100 * v / total, 1) for k, v in sorted(counts.items())})

train {1: 8461, 2: 8340} {1: 50.4, 2: 49.6}
validation {1: 508, 2: 492} {1: 50.8, 2: 49.2}
test {1: 781, 2: 751} {1: 51.0, 2: 49.0}


## Token lengths

Is truncation needed? `story` is O1 + H + O2 without separators, with H = the longer hypothesis.

In [4]:
tok = {s: [{f: word_tokenize(r[f]) for f in FIELDS} | {"label": r["label"]} for r in ds[s]] for s in ["train", "validation"]}


def words(tokens):
    return [t for t in tokens if not is_punct(t)]


def describe(values):
    v = np.array(values)
    return f"mean {v.mean():5.1f}  median {np.median(v):4.0f}  p95 {np.percentile(v, 95):4.0f}  max {v.max():4d}"


def story_length(r):
    return (len(words(r["observation_1"])) + len(words(r["observation_2"]))
            + max(len(words(r["hypothesis_1"])), len(words(r["hypothesis_2"]))))


train = tok["train"]
for field in FIELDS:
    print(f"{field:15s}", describe([len(words(r[field])) for r in train]))
print(f"{'story':15s}", describe([story_length(r) for r in train]))
print("longest story in validation:", max(story_length(r) for r in tok["validation"]))

observation_1   mean   8.2  median    8  p95   13  max   17
observation_2   mean   9.4  median   10  p95   14  max   19
hypothesis_1    mean   8.5  median    8  p95   14  max   30


hypothesis_2    mean   8.5  median    8  p95   14  max   30


story           mean  26.7  median   26  p95   36  max   56
longest story in validation: 50


## Difference between the two hypotheses

How many word edits separate H1 and H2? Does length give away the label?

In [5]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        current = [i]
        for j, y in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (x != y)))
        previous = current
    return previous[-1]


distances = [edit_distance(words(r["hypothesis_1"]), words(r["hypothesis_2"])) for r in train]
print("word edit distance:", describe(distances))
d = np.array(distances)
for limit in [1, 2, 3, 5, 10]:
    print(f"pairs with distance <= {limit:2d}: {100 * (d <= limit).mean():5.1f}%")

correct = [len(words(r["hypothesis_1" if r["label"] == 1 else "hypothesis_2"])) for r in train]
wrong = [len(words(r["hypothesis_2" if r["label"] == 1 else "hypothesis_1"])) for r in train]
print("mean length, correct hypothesis:", round(np.mean(correct), 2), "| wrong hypothesis:", round(np.mean(wrong), 2))
longer = [c > w for c, w in zip(correct, wrong) if c != w]
print("when lengths differ, the longer one is correct:", round(100 * np.mean(longer), 1), "%")
print("pairs with different lengths:", round(100 * np.mean([c != w for c, w in zip(correct, wrong)]), 1), "%")


def pick_shorter_accuracy(rows):
    """Accuracy of always picking the shorter hypothesis; ties count as half right."""
    score = 0.0
    for r in rows:
        h1, h2 = len(words(r["hypothesis_1"])), len(words(r["hypothesis_2"]))
        score += 0.5 if h1 == h2 else float((1 if h1 < h2 else 2) == r["label"])
    return 100 * score / len(rows)


print("pick the shorter hypothesis, train:", round(pick_shorter_accuracy(train), 1), "%")
print("pick the shorter hypothesis, validation:", round(pick_shorter_accuracy(tok["validation"]), 1), "%")

word edit distance: mean   3.4  median    3  p95    9  max   27
pairs with distance <=  1:  28.1%
pairs with distance <=  2:  48.8%
pairs with distance <=  3:  64.7%
pairs with distance <=  5:  83.2%
pairs with distance <= 10:  97.4%
mean length, correct hypothesis: 8.47 | wrong hypothesis: 8.57
when lengths differ, the longer one is correct: 42.9 %
pairs with different lengths: 54.7 %


pick the shorter hypothesis, train: 53.9 %
pick the shorter hypothesis, validation: 53.8 %


## Vocabulary and rare words

In [6]:
counts = collections.Counter(t for r in train for f in FIELDS for t in r[f])
word_counts = collections.Counter({t: n for t, n in counts.items() if not is_punct(t)})
total = sum(word_counts.values())
print("tokens:", sum(counts.values()), "| punctuation tokens:", sum(counts.values()) - total,
      f"({100 * (sum(counts.values()) - total) / sum(counts.values()):.1f}%)")
print("unique words:", len(word_counts))
for limit in [1, 2, 5]:
    rare = [t for t, n in word_counts.items() if n <= limit]
    print(f"words seen <= {limit} times: {len(rare)} ({100 * len(rare) / len(word_counts):.1f}% of unique words, "
          f"{100 * sum(word_counts[t] for t in rare) / total:.2f}% of tokens)")
print("most common:", word_counts.most_common(15))
print("punctuation:", collections.Counter({t: n for t, n in counts.items() if is_punct(t)}).most_common(10))

tokens: 655477 | punctuation tokens: 73846 (11.3%)
unique words: 17768
words seen <= 1 times: 6030 (33.9% of unique words, 1.04% of tokens)
words seen <= 2 times: 8794 (49.5% of unique words, 1.99% of tokens)
words seen <= 5 times: 12082 (68.0% of unique words, 4.08% of tokens)
most common: [('the', 23484), ('to', 23004), ('a', 19733), ('was', 16228), ('and', 13074), ('her', 8933), ('his', 7597), ('I', 7588), ('in', 6519), ('for', 6005), ('of', 5694), ('had', 5617), ("'s", 4798), ('he', 4781), ('on', 4637)]
punctuation: [('.', 65516), (',', 6061), ('!', 1777), ('$', 116), ('``', 102), ("'", 82), ("''", 37), ('’', 32), ('-', 29), ('?', 24)]


## Embedding coverage: GloVe 840B 300d

Only the vocabulary is read here. Share of tokens and unique words with a vector, without punctuation tokens.

In [7]:
vocabulary = set()
with zipfile.ZipFile(EMBEDDINGS_DIR / "glove.840B.300d.zip") as archive:
    with archive.open("glove.840B.300d.txt") as file:
        for line in io.TextIOWrapper(file, encoding="utf-8", errors="replace", newline="\n"):
            vocabulary.add(line.rstrip("\n").rsplit(" ", 300)[0])
print("GloVe words:", len(vocabulary))


def coverage(counter):
    total = sum(counter.values())
    missing = {t: n for t, n in counter.items() if t not in vocabulary}
    return (f"tokens {100 * (1 - sum(missing.values()) / total):6.2f}%  "
            f"unique words {100 * (1 - len(missing) / len(counter)):6.2f}%  (unique: {len(counter)})")


def lowered(counter):
    result = collections.Counter()
    for t, n in counter.items():
        result[t.lower()] += n
    return result


for split in ds:
    c = collections.Counter(t for r in ds[split] for f in FIELDS for t in word_tokenize(r[f]) if not is_punct(t))
    print(f"{split:11s} NLTK, original case:", coverage(c))
    if split == "train":
        print(f"{split:11s} NLTK, lowercased:   ", coverage(lowered(c)))

missing = collections.Counter({t: n for t, n in word_counts.items() if t not in vocabulary})
print("most common missing:", missing.most_common(25))

GloVe words: 2196017


train       NLTK, original case: tokens  99.93%  unique words  98.03%  (unique: 17768)
train       NLTK, lowercased:    tokens  99.90%  unique words  97.49%  (unique: 16189)


validation  NLTK, original case: tokens  99.93%  unique words  99.47%  (unique: 4179)


test        NLTK, original case: tokens  99.94%  unique words  99.42%  (unique: 5676)
most common missing: [('A+', 6), ('Fariah', 5), ('Shalil', 4), ('Marisse', 4), ('Tanice', 4), ('BunnBunn', 4), ('Loraline', 4), ('Benisha', 4), ('forgeted', 3), ('inessentialed', 3), ('Trichelle', 3), ('Eerin', 3), ('Tuie', 3), ('SnowGirl', 3), ('Lamron', 3), ('Kenden', 3), ('Aurella', 2), ('non-beloved', 2), ('Tabathas', 2), ("did't", 2), ('non-watchfuled', 2), ('non-sowned', 2), ('grubhub', 2), ('Eddit', 2), ('indefinited', 2)]


## Effect of other preprocessing on coverage

A tokenizer that only splits on whitespace, stemming, and whether important tokens have vectors.

In [8]:
whitespace = collections.Counter(t for r in ds["train"] for f in FIELDS for t in r[f].split())
print("whitespace split only:", coverage(whitespace))

stemmer = PorterStemmer()
stemmed = collections.Counter()
for t, n in word_counts.items():
    stemmed[stemmer.stem(t)] += n
print("Porter stemmed:       ", coverage(stemmed))
examples = ["family", "happily", "because", "moved", "decided", "lovely"]
print({w: (stemmer.stem(w), stemmer.stem(w) in vocabulary) for w in examples})

print({t: t in vocabulary for t in ["not", "n't", "because", "alot", "'s", "'m", "<unk>", "<pad>", "<sep>"]})
print("negation tokens in train:", {t: word_counts[t] for t in ["not", "n't", "never", "no"]})

whitespace split only: tokens  94.98%  unique words  63.22%  (unique: 27735)
Porter stemmed:        tokens  98.30%  unique words  81.15%  (unique: 11258)
{'family': ('famili', True), 'happily': ('happili', False), 'because': ('becaus', True), 'moved': ('move', True), 'decided': ('decid', True), 'lovely': ('love', True)}
{'not': True, "n't": True, 'because': True, 'alot': True, "'s": True, "'m": True, '<unk>': True, '<pad>': False, '<sep>': False}
negation tokens in train: {'not': 2099, "n't": 3211, 'never': 1121, 'no': 677}
